In [1]:
import pyodbc

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=.\SAGE100;"
    "DATABASE=Test;"
    "UID=sa;"
    "PWD=123456;"
    "TrustServerCertificate=yes;"
)

try:
    conn = pyodbc.connect(conn_str)
    print("Connexion réussie")
except Exception as e:
    print(e)

Connexion réussie


In [2]:
# ============================================================
# 01_exploration.ipynb
# Exploration des données de stock SAGE
# Changer BASE_NAME pour analyser une autre base 
# ============================================================

# ── Cellule 1 : Configuration ────────────────────────────────
BASE_NAME = 'DEMO_PHARMA'   # ← changer ici pour STE_NGDM ou autre base pour l'analyse 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sqlalchemy import create_engine, text
import pyodbc
import warnings
warnings.filterwarnings('ignore')

# Style global
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family']       = 'DejaVu Sans'
sns.set_palette('tab10')

print(f"✅ Configuration chargée — Base cible : {BASE_NAME}")

✅ Configuration chargée — Base cible : DEMO_PHARMA


In [3]:
from sqlalchemy import create_engine, text

SERVER   = r'SALMAIKSOD\SAGE100'
DATABASE = 'Test'
USERNAME = 'sa'
PASSWORD = '123456'

# SQLAlchemy avec SQL Auth
conn_str = (
    f"mssql+pyodbc://{USERNAME}:{PASSWORD}@{SERVER}/{DATABASE}"
    f"?driver=ODBC+Driver+17+for+SQL+Server"
    f"&TrustServerCertificate=yes"
)

engine = create_engine(conn_str, fast_executemany=True)

# Test
with engine.connect() as conn:
    r = conn.execute(text("SELECT DB_NAME() AS db")).fetchone()
    print(f"✅ Connexion SQLAlchemy OK — Base active : {r.db}")

✅ Connexion SQLAlchemy OK — Base active : Test


In [4]:
import os
os.makedirs('../outputs', exist_ok=True)

query = text("""
SELECT 
    BaseName, DateJour, AR_Ref, AR_Design,
    FA_CodeFamille, FA_Intitule,
    CL_No1, CL_Intitule1,
    DE_No, DE_Intitule,
    TotalEntree, TotalSortie,
    ValeurEntree, ValeurSortie,
    StockFinal, ValeurFinale
FROM Test.stock.VW_StockJoursAvecMvt
WHERE BaseName = :base
ORDER BY AR_Ref, DE_No, DateJour
""")

with engine.connect() as conn:
    df_raw = pd.read_sql(query, conn, params={'base': BASE_NAME})

# Types
df_raw['DateJour']     = pd.to_datetime(df_raw['DateJour'])
df_raw['DE_No']        = pd.to_numeric(df_raw['DE_No'],        errors='coerce').fillna(0).astype(int)
df_raw['TotalEntree']  = pd.to_numeric(df_raw['TotalEntree'],  errors='coerce').fillna(0)
df_raw['TotalSortie']  = pd.to_numeric(df_raw['TotalSortie'],  errors='coerce').fillna(0)
df_raw['ValeurEntree'] = pd.to_numeric(df_raw['ValeurEntree'], errors='coerce').fillna(0)
df_raw['ValeurSortie'] = pd.to_numeric(df_raw['ValeurSortie'], errors='coerce').fillna(0)
df_raw['StockFinal']   = pd.to_numeric(df_raw['StockFinal'],   errors='coerce').fillna(0)
df_raw['ValeurFinale'] = pd.to_numeric(df_raw['ValeurFinale'], errors='coerce').fillna(0)

print(f"✅ Données chargées")
print(f"   Lignes           : {len(df_raw):,}")
print(f"   Articles uniques : {df_raw['AR_Ref'].nunique():,}")
print(f"   Dépôts uniques   : {df_raw['DE_No'].nunique():,}")
print(f"   Période          : {df_raw['DateJour'].min().date()} → {df_raw['DateJour'].max().date()}")
print(f"   Jours couverts   : {df_raw['DateJour'].nunique():,}")
df_raw.head(3)

✅ Données chargées
   Lignes           : 25,316
   Articles uniques : 32
   Dépôts uniques   : 3
   Période          : 2022-01-01 → 2025-05-01
   Jours couverts   : 1,217


,BaseName,DateJour,AR_Ref,AR_Design,FA_CodeFamille,FA_Intitule,CL_No1,CL_Intitule1,DE_No,DE_Intitule,TotalEntree,TotalSortie,ValeurEntree,ValeurSortie,StockFinal,ValeurFinale
0,DEMO_PHARMA,2022-01-01,GENE-001,Amoxicilline 500mg gél bt12,GENE,Médicaments génériques,1,Voies respiratoires,1,Entrepôt Central Agadir,1000.0,0.0,22000.0,0.00,1000.0,22000.00
1,DEMO_PHARMA,2022-01-02,GENE-001,Amoxicilline 500mg gél bt12,GENE,Médicaments génériques,1,Voies respiratoires,1,Entrepôt Central Agadir,0.0,67.0,0.0,1577.18,933.0,20422.82
2,DEMO_PHARMA,2022-01-03,GENE-001,Amoxicilline 500mg gél bt12,GENE,Médicaments génériques,1,Voies respiratoires,1,Entrepôt Central Agadir,0.0,50.0,0.0,1441.00,883.0,18981.82
